In [7]:
from darts import TimeSeries, datasets
from sktime.forecasting.base import ForecastingHorizon
import warnings
from darts.utils.missing_values import fill_missing_values
from sktime.performance_metrics.forecasting import mean_absolute_error
from sktime.forecasting.timesfm import TimesFMForecaster
from darts.utils.missing_values import fill_missing_values
from sktime.performance_metrics.forecasting import mean_absolute_error, MeanAbsoluteScaledError, mean_absolute_percentage_error
from sktime.forecasting.timesfm2 import TimesFM2Forecaster


In [8]:
forecaster = TimesFM2Forecaster(device_map="cuda")


In [9]:
import pickle

with open("darts_univariate_catalog_ordered.pkl", "rb") as f:
    data = pickle.load(f)

data

,dataset_name,load_variable,seasonal,split,test_prediction_split,has_missing_values,length,freq
0,AusBeer,((( Y\ndate \n1956-...,True,207.0,4,False,211,<QuarterBegin: startingMonth=10>
1,Wooly,((( Y\ndate \n196...,True,115.0,4,False,119,<QuarterBegin: startingMonth=10>
2,MonthlyMilk,((( Pounds per cow\nMonth ...,True,156.0,12,False,168,<MonthBegin>
3,AirPassengers,((( #Passengers\nMonth ...,True,132.0,12,False,144,<MonthBegin>
4,Sunspots,((( Sunspots\nMonth \...,True,2808.0,12,False,2820,<MonthBegin>
5,MonthlyMilkIncomplete,((( Pounds per cow\nMonth ...,True,156.0,12,True,168,<MonthBegin>
6,Wine,((( Y\ndate \n1...,True,164.0,12,False,176,<MonthBegin>
7,ETTh2_OT,((( OT\ndate ...,True,17396.0,24,False,17420,<Hour>
8,ETTh1_OT,((( OT\ndate \n201...,True,17396.0,24,False,17420,<Hour>
9,TaxiNewYork,((( #Passengers\ntime ...,True,10272.0,48,False,10320,<30 * Minutes>


In [10]:
results_timesfm = []
Mase_con = MeanAbsoluteScaledError()

In [11]:
import os
import matplotlib.pyplot as plt


def plot_forecast(y_train, y_test, y_pred, dataset_name, save_dir="timesfm"):

    # Create folder
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(12, 5))

    # Training data
    plt.plot(
        y_train.time_index,
        y_train.values().flatten(),
        label="Train",
        linewidth=1.5
    )

    # Actual test data
    plt.plot(
        y_test.time_index,
        y_test.values().flatten(),
        label="Test",
        linewidth=2
    )

    # Chronos prediction
    plt.plot(
        y_pred.index,
        y_pred.iloc[:, 0].values,
        label="Chronos Prediction",
        linewidth=2,
        linestyle="--"
    )

    plt.title(f"timesfm Forecast - {dataset_name}")
    plt.xlabel("Time")
    plt.ylabel("Value")

    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Make filename Windows-safe
    safe_name = "".join(
        c if c.isalnum() or c in (" ", "_", "-") else "_"
        for c in str(dataset_name)
    ).strip()

    save_path = os.path.join(
        save_dir,
        f"{safe_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    return save_path

In [14]:
for i in range(0,13):
    
    y = data["load_variable"][i]
    
    if data["has_missing_values"][i] == True:
        y = fill_missing_values(y,fill="auto")    ###### HANDLE MISSING VALUES

    split_point = int(data["split"][i]) if data["seasonal"][i] else data["split"][i]
    y_train, y_test = y.split_before(split_point)
    
    fh = ForecastingHorizon(y_test.time_index, is_relative=False)

    forecaster.fit(y_train.to_dataframe())
    y_pred = forecaster.predict(fh)

    dataset_name = data["dataset_name"][i]
    plot_forecast(
        y_train=y_train,
        y_test=y_test,
        y_pred=y_pred,
        dataset_name=dataset_name,
        save_dir="timesfm"
    )


    mae = mean_absolute_error(y_test.to_dataframe(),y_pred)
    mase = Mase_con(y_test.to_dataframe(), y_pred, y_train=y_train.to_dataframe())
    smape = mean_absolute_percentage_error(y_test.to_dataframe(),y_pred, symmetric = True)
    
    new_row = {"data": data["dataset_name"][i],"MAE":mae,"mase":mase,"sMAPE":smape}
    results_timesfm.append(new_row)



c:\Users\Diat\anaconda3\envs\times\lib\site-packages\sktime\performance_metrics\forecasting\_base.py:658: UserWarning: y_pred and y_true do not have the same row index. This may indicate incorrect objects passed to the metric. Indices of y_true will be used for y_pred.
  warn(
c:\Users\Diat\anaconda3\envs\times\lib\site-packages\sktime\performance_metrics\forecasting\_base.py:658: UserWarning: y_pred and y_true do not have the same row index. This may indicate incorrect objects passed to the metric. Indices of y_true will be used for y_pred.
  warn(
c:\Users\Diat\anaconda3\envs\times\lib\site-packages\sktime\performance_metrics\forecasting\_base.py:658: UserWarning: y_pred and y_true do not have the same row index. This may indicate incorrect objects passed to the metric. Indices of y_true will be used for y_pred.
  warn(
c:\Users\Diat\anaconda3\envs\times\lib\site-packages\sktime\performance_metrics\forecasting\_base.py:658: UserWarning: y_pred and y_true do not have the same row inde

In [18]:

for i in range(13,16):
    
    y = data["load_variable"][i]

    if data["has_missing_values"][i] == True:
        y = fill_missing_values(y,fill="auto")    ###### HANDLE MISSING VALUES

    split_point = int(data["split"][i]) if data["seasonal"][i] else data["split"][i]
    y_train, y_test = y.split_before(split_point)
    
    forecaster = TimesFM2Forecaster(device_map="cuda", config= {"horizon_length":len(y_test)})

    fh = ForecastingHorizon(y_test.time_index, is_relative=False)

    forecaster.fit(y_train.to_dataframe())
    y_pred = forecaster.predict(fh)


    plot_forecast(
        y_train=y_train,
        y_test=y_test,
        y_pred=y_pred,
        dataset_name=dataset_name,
        save_dir="timesfm"
    )


    mae = mean_absolute_error(y_test.to_dataframe(),y_pred)
    mase = Mase_con(y_test.to_dataframe(), y_pred, y_train=y_train.to_dataframe())
    smape = mean_absolute_percentage_error(y_test.to_dataframe(),y_pred, symmetric = True)
    
    new_row = {"data": data["dataset_name"][i],"MAE":mae,"mase":mase,"sMAPE":smape}
    results_timesfm.append(new_row)



Loading weights: 0it [00:00, ?it/s]

[transformers] TimesFmModelForPrediction LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
model.layers.{0...19}.post_attention_layernorm.weight   | UNEXPECTED | 
model.layers.{0...19}.self_attn.q_norm.weight           | UNEXPECTED | 
model.layers.{0...19}.input_layernorm.weight            | UNEXPECTED | 
model.layers.{0...19}.pre_feedforward_layernorm.weight  | UNEXPECTED | 
model.layers.{0...19}.mlp.ff1.weight                    | UNEXPECTED | 
model.layers.{0...19}.self_attn.v_proj.weight           | UNEXPECTED | 
model.layers.{0...19}.self_attn.k_norm.weight           | UNEXPECTED | 
model.layers.{0...19}.self_attn.k_proj.weight           | UNEXPECTED | 
model.layers.{0...19}.mlp.ff0.weight                    | UNEXPECTED | 
model.layers.{0...19}.self_attn.q_proj.weight           | UNEXPECTED | 
model.input_ff_layer.output_layer.bias  

Loading weights: 0it [00:00, ?it/s]

[transformers] TimesFmModelForPrediction LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
model.layers.{0...19}.post_attention_layernorm.weight   | UNEXPECTED | 
model.layers.{0...19}.self_attn.q_norm.weight           | UNEXPECTED | 
model.layers.{0...19}.input_layernorm.weight            | UNEXPECTED | 
model.layers.{0...19}.pre_feedforward_layernorm.weight  | UNEXPECTED | 
model.layers.{0...19}.mlp.ff1.weight                    | UNEXPECTED | 
model.layers.{0...19}.self_attn.v_proj.weight           | UNEXPECTED | 
model.layers.{0...19}.self_attn.k_norm.weight           | UNEXPECTED | 
model.layers.{0...19}.self_attn.k_proj.weight           | UNEXPECTED | 
model.layers.{0...19}.mlp.ff0.weight                    | UNEXPECTED | 
model.layers.{0...19}.self_attn.q_proj.weight           | UNEXPECTED | 
model.input_ff_layer.output_layer.bias  

Loading weights: 0it [00:00, ?it/s]

[transformers] TimesFmModelForPrediction LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
model.layers.{0...19}.post_attention_layernorm.weight   | UNEXPECTED | 
model.layers.{0...19}.self_attn.q_norm.weight           | UNEXPECTED | 
model.layers.{0...19}.input_layernorm.weight            | UNEXPECTED | 
model.layers.{0...19}.pre_feedforward_layernorm.weight  | UNEXPECTED | 
model.layers.{0...19}.mlp.ff1.weight                    | UNEXPECTED | 
model.layers.{0...19}.self_attn.v_proj.weight           | UNEXPECTED | 
model.layers.{0...19}.self_attn.k_norm.weight           | UNEXPECTED | 
model.layers.{0...19}.self_attn.k_proj.weight           | UNEXPECTED | 
model.layers.{0...19}.mlp.ff0.weight                    | UNEXPECTED | 
model.layers.{0...19}.self_attn.q_proj.weight           | UNEXPECTED | 
model.input_ff_layer.output_layer.bias  

In [19]:
results_timesfm

[{'data': 'AusBeer',
  'MAE': np.float64(11.425537109375),
  'mase': np.float64(0.20007315917470675),
  'sMAPE': np.float64(0.027514717838660375)},
 {'data': 'Wooly',
  'MAE': np.float64(802.8419189453125),
  'mase': np.float64(1.4294144646920244),
  'sMAPE': np.float64(0.15095574859042293)},
 {'data': 'MonthlyMilk',
  'MAE': np.float64(13.520116170247396),
  'mase': np.float64(0.3479936908648866),
  'sMAPE': np.float64(0.015668343478552768)},
 {'data': 'AirPassengers',
  'MAE': np.float64(34.71576690673828),
  'mase': np.float64(1.4414470569834277),
  'sMAPE': np.float64(0.07403751758617609)},
 {'data': 'Sunspots',
  'MAE': np.float64(22.525901158650715),
  'mase': np.float64(1.8747873948851217),
  'sMAPE': np.float64(0.3303930408579626)},
 {'data': 'MonthlyMilkIncomplete',
  'MAE': np.float64(16.384592692057293),
  'mase': np.float64(0.4314665082006252),
  'sMAPE': np.float64(0.0190353256961211)},
 {'data': 'Wine',
  'MAE': np.float64(2067.6066080729165),
  'mase': np.float64(0.43657

In [20]:
import pandas as pd
timesfm_results = pd.DataFrame(results_timesfm)
timesfm_results.to_pickle("timesfm_results.pkl")